# 06 - Análise do Espaço Latente Pré-Classificador, Avaliação Hierárquica e Auditoria de Sonda Linear

Este notebook reproduz a extração de representações latentes pré-classificador
(2048-d ResNet50 e 768-d ConvNeXt), avaliação de KNN cosseno ($k=5$), Silhouette Score de alta dimensão
com auditoria de suporte de classes, correlações hierárquicas Species $\to$ Genus $\to$ Order (Tabela 6),
auditoria de sonda linear independente em treino/validação (`AD-012`), e projeção qualitativa UMAP 2D (Figura 2).

**Invariantes científicas (`AD-012`):**
- Embeddings extraídos imediatamente antes de `model.fc` via pre-hook sem redução transdutiva de dimensionalidade.
- KNN cosseno ($k=5$) com split 80/20 disjunto para garantir estritamente zero auto-vizinhança.
- Silhouette Score no espaço original de alta dimensão com auditoria de classes com $<2$ amostras.
- UMAP 2D é estritamente uma visualização qualitativa e não expressa distância filogenética ou separação formal de clusters.
- Sonda linear para teste do gargalo de classificação treinada exclusivamente em treino/validação sem vazamento do teste.

In [ ]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
project_root = None
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        project_root = r
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break
if project_root is None:
    raise RuntimeError("Could not resolve repository root containing src/viral_cnn")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    free_gpu_memory,
    reserve_run_directory,
    preflight_latent_inputs,
)
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.models.registry import build_registered_model
from viral_cnn.eval.latent import (
    extract_latent_embeddings,
    _sha256_file,
    EmbeddingArtifact,
    save_embedding_artifact,
    evaluate_cosine_knn,
    evaluate_silhouette_with_audit,
    calculate_taxonomic_correlations,
    LinearProbeClassifier,
    audit_classifier_bottleneck,
    export_umap_projection,
)
from viral_cnn.taxonomy.curation import generate_synthetic_taxonomy, sanitize_hybrid_taxonomy

# 1. Inicialização do runtime e contexto de execução (AD-017)
ctx = setup_notebook_runtime()
if ctx.run_id is None:
    ctx.run_id = "c01-" + __import__("datetime").datetime.now().strftime("%Y%m%dT%H%M%S%f")
output_dir = reserve_run_directory(project_root, ctx.run_id, "06_ictv_latent_space_and_hierarchy")
ctx.output_dir = output_dir

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


In [ ]:
# 2. Carregamento de Taxonomia Saneada e Mapeamento de Ranks
cleaned_tax_path = Path(ctx.topology.paths["short"]["hybrid_taxonomy_parquet"])
if not ctx.is_smoke_test and not cleaned_tax_path.is_file():
    raise FileNotFoundError(f"Taxonomia empírica ausente: {cleaned_tax_path}")
if cleaned_tax_path.is_file():
    print(f"[LOG_PATH] Loading taxonomy from: {cleaned_tax_path.resolve()}")
    taxonomy_df = pd.read_parquet(cleaned_tax_path)
elif ctx.is_smoke_test:
    taxonomy_df = generate_synthetic_taxonomy(num_classes=100)
else:
    raise FileNotFoundError(f"Taxonomia ausente: {cleaned_tax_path}")

print(f"Taxonomia carregada: {len(taxonomy_df)} espécies")
print(f"Genera: {taxonomy_df['Genus'].nunique()}, Orders: {taxonomy_df['Order'].nunique()}")


In [ ]:
# 3. Extração por split com identidade operacional e checkpoint registrado
np.random.seed(ctx.seed)
torch.manual_seed(ctx.seed)
models_to_evaluate = [
    {"name": "resnet_clean_species", "display": "ResNet50", "dim": 2048},
    {"name": "resnet_noise_species", "display": "ResNet50_F", "dim": 2048},
    {"name": "convnext_zero_species", "display": "ConvNeXt_Z", "dim": 2048},
    {"name": "convnext_transfer_species", "display": "ConvNeXt_T", "dim": 2048},
    {"name": "convnext_noise_species", "display": "ConvNeXt_F", "dim": 2048},
]
species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
if ctx.is_smoke_test:
    # Isolated deterministic fixtures: distinct sequence content for each split and run.
    fixture_dir = output_dir / "fixtures"
    fixture_dir.mkdir(parents=True, exist_ok=True)
    split_files = {}
    bases = np.array(list("ACGT"))
    for split_index, split in enumerate(("train", "validation", "test")):
        rng = np.random.default_rng(ctx.seed + split_index)
        sequences = ["".join(rng.choice(bases, size=512)) for _ in range(16)]
        fixture_path = fixture_dir / f"{split}.parquet"
        pd.DataFrame({"sequence_padded": sequences, "label": [i % 5 for i in range(16)]}).to_parquet(fixture_path, index=False)
        split_files[split] = [str(fixture_path.resolve())]
else:
    split_files = {split: resolve_split_files(species_parquet_dir, split) for split in ("train", "validation", "test")}
partition_digests = {str(path): _sha256_file(path) for paths in split_files.values() for path in paths}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    left_hashes = {partition_digests[str(path)] for path in split_files[left]}
    right_hashes = {partition_digests[str(path)] for path in split_files[right]}
    if left_hashes.intersection(right_hashes):
        raise ValueError(f"Partition content identity overlaps between {left} and {right}")
checkpoint_paths = {m["name"]: ctx.topology.paths["checkpoints"].get(m["name"]) for m in models_to_evaluate}
if not ctx.is_smoke_test:
    preflight = preflight_latent_inputs(split_files, taxonomy_df, checkpoint_paths, [m["name"] for m in models_to_evaluate])
else:
    preflight = {"status": "SYNTHETIC_SMOKE", "splits": {k: [str(x) for x in v] for k, v in split_files.items()}}
    if any(not files for files in split_files.values()):
        raise FileNotFoundError("Smoke fixtures must provide distinct train, validation, and test partitions")

species_to_genus = {}
species_to_order = {}
genus_map = {g: i for i, g in enumerate(taxonomy_df["Genus"].unique())}
order_map = {o: i for i, o in enumerate(taxonomy_df["Order"].unique())}
for _, row in taxonomy_df.iterrows():
    label = int(row["model_label"])
    species_to_genus[label] = genus_map[row["Genus"]]
    species_to_order[label] = order_map[row["Order"]]

embedding_artifacts = {}
embeddings_cache, labels_cache, identities_cache = {}, {}, {}
probe_instances, probe_audits = {}, []
if ctx.is_smoke_test:
    class SmokeLatentModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.projection = nn.Linear(512, 2048)
            self.fc = nn.Linear(2048, max(int(taxonomy_df["model_label"].max()) + 1, 1))
        def forward(self, tokens):
            return self.fc(torch.relu(self.projection(tokens.float())))

def embedding_manifest(model_spec, split, files, dataset, count, checkpoint):
    return {
        "verified": True, "variant": model_spec["name"], "split": split,
        "partitions": [str(Path(f).resolve()) for f in files],
        "partition_sha256": [partition_digests[str(f)] for f in files],
        "checkpoint": str(Path(checkpoint).resolve()) if checkpoint and Path(checkpoint).is_file() else "synthetic-smoke-initialization",
        "checkpoint_sha256": _sha256_file(checkpoint) if checkpoint and Path(checkpoint).is_file() else "synthetic",
        "extractor_code_sha256": _sha256_file(Path(__import__("viral_cnn.eval.latent", fromlist=["__file__"]).__file__)),
        "class_order": sorted(int(x) for x in taxonomy_df["model_label"].unique()),
        "encoding": {"alphabet": "A/C/G/T", "channels": ["A", "C", "G", "T"], "length": 512},
        "mode": "smoke" if ctx.is_smoke_test else "empirical",
        "extractor": "viral_cnn.eval.latent.extract_latent_embeddings:v1",
        "population_available": dataset.total_samples, "extracted_count": count,
        "sampling": {"rule": "none", "seed": ctx.seed},
    }

for model_spec in models_to_evaluate:
    name = model_spec["name"]
    checkpoint = checkpoint_paths[name]
    if not ctx.is_smoke_test and not Path(checkpoint).is_file():
        raise FileNotFoundError(f"Required checkpoint for '{name}' not found at: {checkpoint}")
    if ctx.is_smoke_test:
        print(f"[LOG_PATH] Smoke model initialized without checkpoint; expected checkpoint: {checkpoint}")
        model = SmokeLatentModel()
    else:
        print(f"[LOG_PATH] Loading checkpoint for {name} from: {Path(checkpoint).resolve()}")
        model = build_registered_model(name, checkpoint_path=checkpoint, device=ctx.device)
    model = model.to(ctx.device)
    model_spec["dim"] = int(model.fc.in_features)

    # Extract only the data allowed to fit and select the probe first.
    for split in ("train", "validation"):
        files = split_files[split]
        dataset = ParquetBatchIterableDataset(
            files, batch_size=32 if ctx.is_smoke_test else 4096, shuffle=False,
            split=split, include_identity=True, partition_ids=partition_digests,
        )
        loader = create_optimized_dataloader(dataset, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
        feats, labels, row_ids = extract_latent_embeddings(model, loader, device=ctx.device, return_identities=True)
        if not len(feats):
            raise ValueError(f"Empty {split} embedding extraction for {name}")
        missing_taxa = sorted(set(labels.tolist()) - set(species_to_genus))
        if missing_taxa:
            raise ValueError(f"Taxonomy does not cover model label {missing_taxa[0]} used by {split} split")
        artifact = EmbeddingArtifact(feats, labels.astype(np.int64), row_ids,
            embedding_manifest(model_spec, split, files, dataset, len(feats), checkpoint))
        artifact.validate()
        cache_path = save_embedding_artifact(artifact, output_dir / "features")
        print(f"[LOG_PATH] Saved {split} embeddings for {name} to: {cache_path.resolve()}")
        embedding_artifacts[(name, split)] = artifact

    probe = LinearProbeClassifier(in_features=model_spec["dim"], num_classes=max(int(taxonomy_df["model_label"].max()) + 1, 1))
    selection = probe.fit_select_and_seal(
        embedding_artifacts[(name, "train")], embedding_artifacts[(name, "validation")],
        epochs=2 if ctx.is_smoke_test else 10, lr=1e-3, weight_decay=1e-4,
        batch_size=256, seed=ctx.seed, device=ctx.device,
    )
    probe_instances[name] = probe
    probe_audits.append({"model": name, "selection": selection})

    # The test loader and its embeddings are first opened only after selection has sealed the weights.
    test_files = split_files["test"]
    test_dataset = ParquetBatchIterableDataset(
        test_files, batch_size=32 if ctx.is_smoke_test else 4096, shuffle=False,
        split="test", include_identity=True, partition_ids=partition_digests,
    )
    test_loader = create_optimized_dataloader(test_dataset, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
    test_feats, test_labels, test_ids, native_logits = extract_latent_embeddings(
        model, test_loader, device=ctx.device, return_identities=True, return_logits=True)
    if not len(test_feats):
        raise ValueError(f"Empty test embedding extraction for {name}")
    missing_taxa = sorted(set(test_labels.tolist()) - set(species_to_genus))
    if missing_taxa:
        raise ValueError(f"Taxonomy does not cover model label {missing_taxa[0]} used by test split")
    test_artifact = EmbeddingArtifact(
        test_feats, test_labels.astype(np.int64), test_ids,
        embedding_manifest(model_spec, "test", test_files, test_dataset, len(test_feats), checkpoint))
    test_artifact.validate()
    cache_path = save_embedding_artifact(test_artifact, output_dir / "features")
    print(f"[LOG_PATH] Saved sealed-probe test embeddings for {name} to: {cache_path.resolve()}")
    embedding_artifacts[(name, "test")] = test_artifact
    embeddings_cache[name], labels_cache[name], identities_cache[name] = test_feats, test_labels, test_ids
    model_spec["native_test_logits"] = native_logits
    free_gpu_memory(model)
    del model

first_model_name = models_to_evaluate[0]["name"]
print(f"Representações latentes prontas: 5 modelos, {len(embeddings_cache[first_model_name])} linhas de teste")


In [ ]:
# 4. Avaliação de KNN Cosseno (k=5) e Silhouette Score (Tabela 5)
table5_rows = []
per_taxon_metrics_cache = {}
knn_audits = []

for m in models_to_evaluate:
    name = m["name"]
    raw_feats = embeddings_cache[name]
    y_sp = labels_cache[name]
    y_ge = np.array([species_to_genus.get(sp, 0) for sp in y_sp])
    y_ord = np.array([species_to_order.get(sp, 0) for sp in y_sp])
    per_taxon_metrics_cache[name] = {}

    for rank_name, labels in [("Species", y_sp), ("Genus", y_ge), ("Order", y_ord)]:
        knn_res = evaluate_cosine_knn(
            features=raw_feats,
            labels=labels,
            k=5,
            random_state=42,
            taxonomic_rank=rank_name,
            row_ids=identities_cache[name],
        )
        knn_audits.append({
            "model": name, "rank": rank_name, "available": knn_res["available"],
            "k_requested": knn_res["k_requested"], "k_effective": knn_res["k"],
            "reference_ids": knn_res["reference_ids"].tolist(), "query_ids": knn_res["query_ids"].tolist(),
            "neighbor_ids": knn_res.get("neighbor_ids", np.empty((0, 0), dtype=str)).tolist(),
            "identity_overlap_count": knn_res["identity_overlap_count"],
            "self_neighbor_count": knn_res.get("self_neighbor_count", 0),
            "n_classes_total": knn_res["n_classes_total"],
            "n_classes_with_query": knn_res["n_classes_with_query"],
            "n_classes_reference_only": knn_res["n_classes_reference_only"],
            "top5_definition": knn_res["top5_definition"],
        })
        sil_res = evaluate_silhouette_with_audit(
            features=raw_feats,
            labels=labels,
            max_samples=min(len(raw_feats), 200 if ctx.is_smoke_test else 5000),
            random_state=42,
            taxonomic_rank=rank_name,
        )
        if not knn_res["available"]:
            knn_per_cluster = pd.DataFrame(columns=["class_label", "knn_accuracy", "knn_top5_accuracy"])
        else:
            knn_per_cluster = knn_res["per_cluster_metrics"]
        sil_per_cluster = sil_res["per_cluster_metrics"]
        merged_cluster = knn_per_cluster.merge(
            sil_per_cluster[["class_label", "silhouette_score"]],
            on="class_label",
            how="left",
        )
        per_taxon_metrics_cache[name][rank_name] = merged_cluster
        table5_rows.append({
            "Model": m["display"],
            "Rank": rank_name,
            "KNN_Top1": knn_res.get("global_knn_accuracy", np.nan),
            "KNN_Top5": knn_res["global_knn_top5_accuracy"],
            "Silhouette_Score": sil_res["global_silhouette_score"],
            "Silhouette_Valid_Class_Rate": sil_res.get("valid_class_rate", 0.0),
            "Silhouette_Retained_Sample_Rate": sil_res.get("retained_sample_rate", 1.0),
            "KNN_Available": knn_res["available"],
            "KNN_Queries": knn_res["n_query"],
            "KNN_Reference": knn_res["n_reference"],
            "KNN_Reference_Query_ID_Overlap": knn_res["identity_overlap_count"],
            "KNN_Unavailable_Reason": knn_res.get("unavailable_reason"),
            "KNN_Valid_Class_Count": knn_res.get("n_classes_with_query", 0),
            "KNN_Top5_Definition": knn_res["top5_definition"],
            "Silhouette_Actual_Sample_Size": sil_res["actual_sample_size"],
            "Silhouette_Total_Classes": sil_res["total_classes_sampled"],
            "Silhouette_Valid_Classes": sil_res["valid_classes_count"],
            "Silhouette_Excluded_Classes": sil_res["excluded_classes_count"],
            "Silhouette_Sampling_Strategy": sil_res["sampling_strategy"],
        })

table5_df = pd.DataFrame(table5_rows)
table5_path = output_dir / "table5_latent_clustering_metrics.csv"
table5_df.to_csv(table5_path, index=False)
knn_audit_path = output_dir / "knn_identity_audit.json"
knn_audit_path.write_text(json.dumps(knn_audits, indent=2) + "\n")
print("Tabela 5 - Métricas de Espaço Latente por Nível Taxonômico:")
print(table5_df.to_string(index=False))


In [5]:
# 5. Correlações Hierárquicas: Species -> Genus e Genus -> Order (Tabela 6)
table6_rows = []

for m in models_to_evaluate:
    m_metrics = per_taxon_metrics_cache[m["name"]]
    df_sp = m_metrics["Species"].copy()
    df_ge = m_metrics["Genus"].copy()
    df_ord = m_metrics["Order"].copy()

    tax_mapping_df = pd.DataFrame({
        "species": list(species_to_genus.keys()),
        "Genus": list(species_to_genus.values()),
        "Order": [species_to_order.get(k, 0) for k in species_to_genus.keys()],
    })

    sp_genus_corr = calculate_taxonomic_correlations(
        child_df=df_sp,
        parent_df=df_ge,
        taxonomy_df=tax_mapping_df,
        child_tax_col="species",
        parent_tax_col="Genus",
        child_level="Species",
        parent_level="Genus",
        min_children=1 if ctx.is_smoke_test else 3,
    )
    table6_rows.append({
        "Model": m["display"],
        "Transition": "Species -> Genus",
        "Pearson_r": sp_genus_corr["pearson_r_knn"],
        "Spearman_rho": sp_genus_corr["spearman_rho_knn"],
        "N_taxa": sp_genus_corr["n_parents_knn"],
    })

    genus_order_corr = calculate_taxonomic_correlations(
        child_df=df_ge,
        parent_df=df_ord,
        taxonomy_df=tax_mapping_df,
        child_tax_col="Genus",
        parent_tax_col="Order",
        child_level="Genus",
        parent_level="Order",
        min_children=1 if ctx.is_smoke_test else 3,
    )
    table6_rows.append({
        "Model": m["display"],
        "Transition": "Genus -> Order",
        "Pearson_r": genus_order_corr["pearson_r_knn"],
        "Spearman_rho": genus_order_corr["spearman_rho_knn"],
        "N_taxa": genus_order_corr["n_parents_knn"],
    })

table6_df = pd.DataFrame(table6_rows)
table6_path = output_dir / "table6_taxonomic_rank_correlations.csv"
table6_df.to_csv(table6_path, index=False)
print("")
print("Tabela 6 - Correlações Hierárquicas entre Níveis Taxonômicos:")
print(table6_df.to_string(index=False))



Tabela 6 - Correlações Hierárquicas entre Níveis Taxonômicos:
     Model       Transition  Pearson_r  Spearman_rho  N_taxa
  ResNet50 Species -> Genus   0.532939      1.000000       3
  ResNet50   Genus -> Order        NaN           NaN       1
ResNet50_F Species -> Genus   0.332877      0.500000       3
ResNet50_F   Genus -> Order        NaN           NaN       1
ConvNeXt_Z Species -> Genus  -0.063433      0.500000       3
ConvNeXt_Z   Genus -> Order        NaN           NaN       1
ConvNeXt_T Species -> Genus   0.804979      1.000000       3
ConvNeXt_T   Genus -> Order        NaN           NaN       1
ConvNeXt_F Species -> Genus   0.992778      0.866025       3
ConvNeXt_F   Genus -> Order        NaN           NaN       1


In [ ]:
# 6. Auditoria da sonda selada no conjunto test (AD-012)
probe_results = []
for model_spec in models_to_evaluate:
    name = model_spec["name"]
    probe = probe_instances[name]
    test = embedding_artifacts[(name, "test")]
    eval_res = probe.evaluate_artifact(test, device=ctx.device)
    native_pred = np.argmax(model_spec["native_test_logits"], axis=1)
    native_model_accuracy = float(np.mean(native_pred == test.labels))
    probe_results.append({
        "Model": model_spec["display"], "Linear_Probe_Test_Acc": eval_res["top1_accuracy"],
        "Linear_Probe_Macro_F1": eval_res["macro_f1"], "N": eval_res["n_test_samples"],
        "Supported_Classes": eval_res["supported_classes"], "Test_Weights_SHA256": eval_res["weights_sha256"],
        "Test_Row_IDs": json.dumps(eval_res["row_ids"]), "Native_Head_Accuracy": native_model_accuracy,
        "Native_Head_Comparison": "same identified test examples",
    })
probe_df = pd.DataFrame(probe_results)
probe_path = output_dir / "linear_probe_audit.csv"
probe_df.to_csv(probe_path, index=False)
(output_dir / "linear_probe_selection.json").write_text(json.dumps(probe_audits, indent=2, default=str) + "\n")
print(probe_df.drop(columns=["Test_Row_IDs"]).to_string(index=False))


In [7]:
# 7. Projeção Qualitativa UMAP 2D (Figura 2)
umap_feats = embeddings_cache["convnext_noise_species"]
umap_y = labels_cache["convnext_noise_species"]
umap_df = export_umap_projection(
    features=umap_feats[:100],
    labels=umap_y[:100],
    output_path=output_dir / "umap_projection_convnext_f.csv",
    random_state=42,
)
print("")
print("[UMAP] Projeção UMAP 2D gerada com sucesso.")
print("AVISO CIENTÍFICO (AD-012): A projeção UMAP 2D destina-se estritamente à visualização qualitativa")
print("e NÃO demonstra distância filogenética ou separação formal de clusters.")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



[UMAP] Projeção UMAP 2D gerada com sucesso.
AVISO CIENTÍFICO (AD-012): A projeção UMAP 2D destina-se estritamente à visualização qualitativa
e NÃO demonstra distância filogenética ou separação formal de clusters.


In [ ]:
# 8. Emissão do Manifesto de Execução Rastreável
manifest = write_notebook_manifest(
    notebook_id="06_ictv_latent_space_and_hierarchy",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    output_json_path=output_dir / "manifest.json",
    run_id=ctx.run_id,
    input_files=[str(cleaned_tax_path)] + [str(path) for paths in split_files.values() for path in paths],
    checkpoints=checkpoint_paths,
    outputs=[str(path) for path in output_dir.iterdir() if path.is_file() and path.name != "manifest.json"],
    configuration={"probe_protocol": "c01_train_val_frozen_v1", "probe_epochs": 2 if ctx.is_smoke_test else 10, "probe_lr": 1e-3, "probe_weight_decay": 1e-4, "probe_batch_size": 256, "probe_seed": ctx.seed, "knn_k": 5, "knn_test_reference_fraction": 0.8, "genome_independence": "unknown"},
    metrics_summary={
        "models_evaluated": len(models_to_evaluate),
        "table5_entries": len(table5_df),
        "table6_entries": len(table6_df),
        "umap_exported": True,
        "probe_variants": len(probe_df),
        "preflight": preflight,
        "genome_independence": "unknown: genome IDs unavailable for test fragments",
    },
)
print("\nManifesto de Execução salvo com sucesso:")
print(json.dumps(manifest.metrics_summary, indent=2))
